# 14 — A2A: agents as peers

**What you'll learn**

- Why A2A is a different protocol from MCP: MCP wires an agent to its *tools*; A2A (Agent2Agent) lets one organization's agent call another's as a *peer*, over the wire
- How discovery works — an **Agent Card** served at the well-known URL `/.well-known/agent-card.json`, advertising a peer's skills and where to reach it
- The A2A **task lifecycle**, including the `input-required` state a peer enters to pause and ask for more input mid-task — first built by hand, then driven with the real `a2a-sdk`
- Resolving a peer's card with `A2ACardResolver`, sending a message through `ClientFactory`, and walking the streamed task events from `submitted` to `completed`
- Larkspur calling a supplier agent (Northwind Supply) mid-shift for a restock ETA — a cross-organization call folded straight into a reorder decision

*Time: ~4 min on a first live run; under a minute cached. Cost: ~$0.01 (one small model call). Cached reruns are free.*

> **Before running this notebook:** `pip install -e ".[a2a]"` (once). It pulls in the official `a2a-sdk` and a small ASGI server stack (`uvicorn`, `starlette`) — the peer agent and its client both run on localhost. Everything else stays the same.

## Two protocols, two directions

Part 4 is about letting the ops desk talk to the outside world, and that has two directions. Chapter 13 pointed the agent *down* at its tools: the [Model Context Protocol](https://modelcontextprotocol.io/specification/2026-07-28) is how an agent discovers and calls the resources, prompts, and tools a server exposes — agent-to-tools. A2A points *sideways*. The [Agent2Agent protocol](https://a2a-protocol.org/v1.0.0/specification/) is how one organization's agent talks to another's as a peer: no shared codebase, no local function call, just an HTTP endpoint that publishes what it can do and a task lifecycle for getting it done. Larkspur Outfitters is a retailer, not a manufacturer; when stock runs low it has to ask a *supplier* — a different company, running its own agent — how fast it can restock. That call is A2A.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## The shape, by hand

Before the SDK does it for us, let's build the smallest thing that is recognizably A2A, so the moving parts stay visible. Two pieces: a **discovery document** and a **task endpoint**. The A2A spec says a peer publishes its Agent Card at a fixed well-known path — [`/.well-known/agent-card.json`](https://a2a-protocol.org/v1.0.0/specification/) — so a caller who knows only the host can find out who lives there and what they do. The task endpoint is where the work happens, and the interesting part of the lifecycle is that a task need not finish in one shot: if the peer needs more information it parks the task in `input-required` and asks. Here is that whole shape in a hand-written `http.server` — a card, and a task function that asks for a warehouse region when the caller forgets it.

In [ ]:
import json, socket, threading, urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

WELL_KNOWN = "/.well-known/agent-card.json"        # the spec-mandated discovery path

TOY_CARD = {
    "name": "Toy Supplier",
    "description": "A hand-built A2A peer that quotes a restock lead time.",
    "version": "0.1.0",
    "skills": [{"id": "restock_quote", "name": "Restock lead time by region"}],
}

TOY_TASKS = {}                                     # task_id -> {"region": ...}

def toy_lifecycle(task_id, text):
    """One task turn -> (task_id, state, message). No region yet -> input-required."""
    region = next((r for r in ("west", "central", "east") if r in text.lower()), None)
    if task_id is None:
        task_id = f"task-{len(TOY_TASKS) + 1}"
        TOY_TASKS[task_id] = {"region": region}
    elif region:
        TOY_TASKS[task_id]["region"] = region
    region = TOY_TASKS[task_id]["region"]
    if region is None:
        return task_id, "input-required", "Which warehouse region -- west, central, east?"
    return task_id, "completed", f"Restock lead time from {region}: 7 business days."

The card is just a dict and the lifecycle is just a function — nothing about A2A is magic. Now wrap them in an HTTP server (a `GET` on the well-known path returns the card, a `POST` runs one task turn) and start it on a free port in a daemon thread so this notebook never blocks. We stop it a few cells down.

In [ ]:
class ToyHandler(BaseHTTPRequestHandler):
    def _send(self, code, obj):
        body = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        self._send(200, TOY_CARD) if self.path == WELL_KNOWN else self._send(404, {})

    def do_POST(self):
        n = int(self.headers.get("Content-Length", 0))
        req = json.loads(self.rfile.read(n) or b"{}")
        tid, state, msg = toy_lifecycle(req.get("task_id"), req.get("text", ""))
        self._send(200, {"task_id": tid, "state": state, "message": msg})

    def log_message(self, *a):
        pass                                       # keep the notebook output quiet

with socket.socket() as s:                         # ask the OS for a free port
    s.bind(("127.0.0.1", 0))
    toy_port = s.getsockname()[1]
toy_server = ThreadingHTTPServer(("127.0.0.1", toy_port), ToyHandler)
threading.Thread(target=toy_server.serve_forever, daemon=True).start()
toy_base = f"http://127.0.0.1:{toy_port}"
print("toy A2A peer serving on", toy_base)

Now be the caller. Discovery first: fetch the card to learn the peer's name and skills. Then drive a task — ask about a SKU *without* naming a region, so the peer pauses at `input-required`, then answer with the region to push the same task to `completed`.

In [ ]:
def toy_get(path):
    with urllib.request.urlopen(toy_base + path, timeout=5) as r:
        return json.loads(r.read())

def toy_post(payload):
    req = urllib.request.Request(toy_base + "/tasks", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())

card = toy_get(WELL_KNOWN)
print("discovered:", card["name"], "->", card["skills"][0]["id"])

turn1 = toy_post({"text": "restock lead time for LK-1015?"})
print("turn 1 ->", turn1["state"], "|", turn1["message"])

turn2 = toy_post({"task_id": turn1["task_id"], "text": "the west warehouse"})
print("turn 2 ->", turn2["state"], "|", turn2["message"])

> **What you should see:** discovery returns `Toy Supplier -> restock_quote` straight from the well-known path. Turn 1 comes back `input-required` with a question about the region, because the caller never named one. Turn 2 supplies `west` against the *same* `task_id`, and the peer drives that task to `completed`. That two-turn pause-and-resume is the whole point of the lifecycle — the shape the real SDK formalizes.

In [ ]:
toy_server.shutdown()
toy_server.server_close()
print("toy peer stopped")

## The real thing: the `a2a-sdk`

The toy got the shape right but skipped everything that makes A2A interoperable: a typed Agent Card, JSON-RPC framing, streamed status events, a task store that persists across turns. The official [`a2a-sdk`](https://github.com/a2aproject/a2a-python) supplies all of it. Our peer is `a2a_servers/supplier_agent.py` — **Northwind Supply**, a fictional wholesaler whose one skill is quoting restock lead times for Larkspur SKUs. It is protocol-complete: it serves a real Agent Card, implements the task lifecycle, and does the same region pause we just hand-wrote, only through the SDK's `TaskUpdater`. We start it in a uvicorn daemon thread (the module's `serve_in_background` context manager) and stop it in the teardown cell.

In [ ]:
import httpx
from a2a_servers.supplier_agent import serve_in_background
from a2a.client import A2ACardResolver, ClientFactory, ClientConfig

supplier = serve_in_background()                   # uvicorn in a daemon thread, on a free port
supplier_base = supplier.__enter__()               # started here; stopped in the teardown cell

hx = httpx.AsyncClient(timeout=10.0)
card = await A2ACardResolver(httpx_client=hx, base_url=supplier_base).get_agent_card()
print("resolved card:", card.name, "| skill:", card.skills[0].id)
print("served at:", supplier_base + "/.well-known/agent-card.json")

The card is resolved, so we know where to POST and which protocol the peer speaks. Build a client from it with `ClientFactory`, then send a message. A2A streams a task as a series of events — the `Task` itself, then `status_update`s as it moves through the lifecycle — so `send_message` returns an async iterator. This helper walks that stream and returns the final state, the task and context ids (needed to resume), and the peer's last message.

In [ ]:
from a2a.helpers import new_text_message, get_stream_response_text
from a2a.types import SendMessageRequest, TaskState, Role

client = ClientFactory(ClientConfig(httpx_client=hx, streaming=True)).create(card)

async def ask_supplier(text, task_id=None, context_id=None):
    """Send one message; walk the streamed events; return (task_id, context_id, state, answer)."""
    msg = new_text_message(text, role=Role.ROLE_USER, task_id=task_id, context_id=context_id)
    state = None
    answer = ""
    async for ev in client.send_message(SendMessageRequest(message=msg)):
        kind = ev.WhichOneof("payload")
        if kind == "task":
            state, task_id, context_id = ev.task.status.state, ev.task.id, ev.task.context_id
        elif kind == "status_update":
            u = ev.status_update
            state, task_id, context_id = u.status.state, u.task_id, u.context_id
        answer = get_stream_response_text(ev) or answer
    return task_id, context_id, TaskState.Name(state), answer

In [ ]:
tid, cid, state1, msg1 = await ask_supplier("What is the restock lead time for LK-1015?")
print("turn 1 ->", state1)
print("   ", msg1)

tid, cid, state2, msg2 = await ask_supplier("west", task_id=tid, context_id=cid)
print("turn 2 ->", state2)
print("   ", msg2)

> **What you should see:** turn 1 ends at `TASK_STATE_INPUT_REQUIRED` — the same pause as the toy, now a named lifecycle state — with Northwind asking which region. Turn 2 answers `west` against the same `task_id`/`context_id`, and the task reaches `TASK_STATE_COMPLETED` carrying a concrete quote: a wholesale price, on-hand units, and a lead time in business days. The numbers are deterministic (derived from the SKU), so a cached rerun is identical.

## Larkspur calls the peer mid-shift

That was the protocol in isolation. Here is why the ops desk cares. During a normal shift the agent notices a SKU sitting at or below its reorder threshold — a local fact, straight from `check_inventory`. To decide whether to reorder it needs something Larkspur does not know: the supplier's lead time and wholesale price. That lives in another company's system, reachable only over A2A. So the ops agent makes the cross-org call — this time naming the region up front, so the task completes in one turn — and folds the peer's answer into a restock recommendation.

One contrast with chapter 13 is worth naming. There the payoff dropped MCP tools into `run_agent` and let the *loop* choose when to call them; here the reorder step is a single known peer call, so we make it directly instead of dressing it as a loop tool. That is the honest shape of most A2A: a peer call is a delegation to another organization, not one more entry in the model's tool menu. Nothing stops you from wrapping an A2A skill as a `shoplab.tools.Tool` behind a sync bridge like chapter 13's `MCPBridge` — do that when you want the model itself to decide to reach for a peer.

In [ ]:
import shoplab.llm
from shoplab import world

low = next(p for p in world.load_products() if p["stock"] <= p["reorder_threshold"])
print(f"low stock: {low['sku']} ({low['name']}) -- {low['stock']} on hand, "
      f"threshold {low['reorder_threshold']}")

_, _, state, quote = await ask_supplier(f"Restock ETA for {low['sku']} from the west warehouse?")
print("supplier:", state)
print("   ", quote)

reco = shoplab.llm.llm(
    f"Larkspur on-hand stock for {low['sku']} ({low['name']}) is {low['stock']} units; "
    f"the reorder threshold is {low['reorder_threshold']}. A supplier peer replied over A2A: "
    f'"{quote}" In ONE sentence, recommend whether to reorder now and roughly how many units.',
    system="You are the Larkspur Outfitters ops desk. Be concise and concrete.",
    max_tokens=120)
print("\nreorder recommendation:")
print(reco.strip())

> **What you should see:** `check_inventory`'s local view (stock at or under threshold) meets Northwind's remote quote, and the model turns the pair into a one-line reorder call. The point is not the exact sentence — it is that a fact from *another organization's agent* arrived over A2A and landed inside a Larkspur decision, with no shared code between the two.

In [ ]:
supplier.__exit__(None, None, None)                # stop the daemon-thread server, join it
await hx.aclose()                                  # close the shared HTTP client
print("Northwind Supply stopped")

## The task lifecycle, named

Both demos walked the same states; A2A gives them names. A task is `submitted` when accepted, `working` while the peer runs, and either `completed`, `failed`, or `canceled` when it ends — with `input-required` as the pause we leaned on, entered when the peer needs the caller to say more before it can finish. The `a2a-sdk` spells these as protobuf enums (`TASK_STATE_INPUT_REQUIRED`, `TASK_STATE_COMPLETED`, and so on), which is why `TaskState.Name(...)` prints the upper-cased form.

| Lifecycle state | SDK enum | Meaning |
|---|---|---|
| `submitted` | `TASK_STATE_SUBMITTED` | the peer accepted the task but has not started |
| `working` | `TASK_STATE_WORKING` | the peer is processing |
| `input-required` | `TASK_STATE_INPUT_REQUIRED` | paused, waiting for the caller's next message on the same task |
| `completed` | `TASK_STATE_COMPLETED` | finished; the result is in the final message |
| `failed` | `TASK_STATE_FAILED` | the peer errored out |
| `canceled` | `TASK_STATE_CANCELED` | the task was canceled before finishing |

The [A2A specification](https://a2a-protocol.org/v1.0.0/specification/) is the authority on these transitions — including the `input-required` state a peer uses to request more input mid-processing.

## One protocol, consolidating

A2A is not the only agent-interoperability protocol, but the field is converging on it. IBM's Agent Communication Protocol (ACP) [merged into A2A under the Linux Foundation](https://lfaidata.foundation/communityblog/2025/08/29/acp-joins-forces-with-a2a-under-the-linux-foundations-lf-ai-data/) in 2025, folding two efforts into one governed standard. For the ops desk that means the peer-to-peer pattern you just drove — discover a card, open a task, resume through `input-required` — is a bet on a stabilizing target, not a moving one.

## Recap

| Concept | One-liner |
|---|---|
| MCP vs A2A | MCP wires an agent to its tools (down); A2A connects agents as peers (sideways). |
| Agent Card | a discovery document at `/.well-known/agent-card.json` — name, skills, where to POST. |
| Well-known path | the fixed URL the A2A spec mandates, so a caller who knows the host finds the card. |
| Task lifecycle | `submitted -> working -> completed`, with `input-required` and `failed`/`canceled` branches. |
| `input-required` | the pause: the peer asks for more input and resumes the *same* task on the next message. |
| Toy peer | a hand-built `http.server` — a card dict and a task function — to see the shape without the SDK. |
| `a2a-sdk` | the real thing: typed cards, JSON-RPC, streamed events, `A2ACardResolver` + `ClientFactory`. |
| `ask_supplier` | walks the streamed events; returns final state, task/context ids, and the peer's message. |
| Cross-org call | a supplier's lead time arrives over A2A and lands inside a Larkspur reorder decision. |
| ACP into A2A | IBM's ACP merged into A2A under the Linux Foundation — the field consolidating on one standard. |

## Exercises

1. **Add a second skill to Northwind's card.** In `a2a_servers/supplier_agent.py`, define a new `AgentSkill` (say, a bulk-discount quote) alongside `restock_quote`, add it to the card's `skills`, and re-resolve the card with `A2ACardResolver`. Confirm the client now sees two skills. What did the *caller* have to change to discover the new capability — and what does that tell you about why discovery lives in the card rather than in the client?
2. **Give the toy a second `input-required` turn.** Extend `toy_lifecycle` so that after it has a region it also needs a quantity: a bare region should leave the task in `input-required` asking "how many units?", and only a region *and* a quantity should reach `completed`. How many round-trips does a caller who supplies nothing up front now make, and how does storing partial state in `TOY_TASKS` keep each turn on the same task?
3. **Handle a failed task on the client.** Make Northwind return `TASK_STATE_FAILED` for an unknown region instead of quoting, then teach the caller of `ask_supplier` to branch on the returned state: retry with a valid region on `failed`, accept the answer on `completed`. Which states should a robust client treat as terminal, and which as a prompt to send another message?

**Next up:** Part 5 — meeting the frameworks. You have built the machinery by hand across four parts; now we hold it up against the libraries the field actually reaches for, and see which pieces they package and which they leave to you.